In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString, MultiPoint, MultiPolygon, MultiLineString

In [ ]:
road = LineString([(80.20,13.00),(80.25,13.05),(80.30,13.10)])
road.geom_type

In [ ]:
import matplotlib.pyplot as plt
x,y = road.xy
plt.plot(x,y,color = 'black', linewidth = 5)
plt.scatter(x,y,color = 'Red')
plt.show()

In [ ]:
zone = Polygon([(0,0), (4,1), (6,4),
                (4,7), (1,6), (-1,3)])



In [ ]:
x,y = zone.exterior.xy
plt.plot(x,y,color = 'black', linewidth = 3)
plt.scatter(x,y,color = 'Red')
plt.grid(True)
plt.title("Curved Road")
plt.show()

In [ ]:
multi  = MultiPolygon([
    Polygon([(0,0),(4,0),(4,4),(0,4)]),
    Polygon([(6,0), (10,0), (10,3), (6,3)])
])

In [ ]:
fig, ax = plt.subplots()
for polygon in multi.geoms:
    x,y = polygon.exterior.xy
    ax.fill(x,y, alpha = 0.5, edgecolor = 'black')

ax.grid(True)
plt.show()

In [ ]:
# irregular

triangle = Polygon([(0,0), (4,0), (2,3)])
concave = Polygon([(0,0),(6,0),(6,2),(3,2),
                  (3,5),(6,4),(6,7),(0,7)])
irregular = Polygon([(0,0),(4,1),(6,4),(4,7),
                    (1,6),(-1,3)])

poly = [triangle, concave, irregular ]

fig, ax = plt.subplots(figsize = (8,6))

for p in poly:
    x,y = p.exterior.xy
    ax.fill(x,y,alpha = 0.5, edgecolor = 'black', linewidth =2)
plt.show()
                       

In [ ]:
triangle = Polygon([(0,0), (4,0), (2,3)])
concave = Polygon([(0,0),(6,0),(6,2),(3,2),
                  (3,5),(6,4),(6,7),(0,7)])
irregular = Polygon([(0,0),(4,1),(6,4),(4,7),
                    (1,6),(-1,3)])

poly = [triangle, concave, irregular ]
color = ['red', 'blue', 'pink']
fig, ax = plt.subplots(figsize = (8,6))

for p,c in zip(poly, color):
    x,y = p.exterior.xy
    ax.fill(x,y,alpha = 0.5,facecolor = c, edgecolor = 'black', linewidth =2)
plt.show()

In [ ]:
# holes on shapes

outer = [(0,0), (10,0), (10,10), (0,10)]
hole = [(3,3), (7,3), (7,7), (3,7)]

# varia 1
p1 = Polygon(outer, [hole])

# varia 2
p2 = Polygon(outer, [
    [(1,1),(2,1),(2,2),(1,2)],
    [(6,6),(8,6),(8,8), (6,8)]
])

# plot
fig, axes = plt.subplots(1,2,figsize = (10,5))

# var 1
x,y = p1.exterior.xy
axes[0].fill(x,y,color = "Skyblue", edgecolor = 'black')

for interior in p1.interiors:
    x,y = zip(*interior.coords)
    axes[0].fill(x,y,color = "white", edgecolor = 'red')


# var 2
x,y = p2.exterior.xy
axes[1].fill(x,y,color = "Skyblue", edgecolor = 'black')

for interior in p2.interiors:
    x,y = zip(*interior.coords)
    axes[1].fill(x,y,color = "white", edgecolor = 'red')


plt.tight_layout()
plt.show()

In [ ]:
# plot polygon's vertices and centroid

x,y = polygon.exterior.xy
fig,ax = plt.subplots(figsize = (8,7))
ax.fill(x,y,alpha = 0.4)
ax.plot(x,y)

c= polygon.centroid
ax.scatter(c.x, c.y, s=100)

for i, (x1,y1) in enumerate(polygon.exterior.coords[:-1]):
    ax.scatter(x1,y1)
    ax.annotate(f"P{i+1}", (x1,y1),xytext=(5,5), textcoords="offset points")
ax.set_aspect("equal")
plt.show()

## CSV to GeoDF


In [ ]:
df = pd.DataFrame({
    "city":["Chennai","Bengaluru","Hyderabad"],
    "latitude":[13.0827,12.9716,17.3850],
    "longitude":[80.2707,77.5946,78.4867],
    "population":[7000000,13000000,10000000]
})

df.to_csv("cities.csv", index = False)

In [ ]:
data = [{
    "name":"sq_hole",
    "outer":"[(0,0),(10,0),(10,10),(0,10)]",
    "holes":"[[(3,3), (7,3), (7,7), (3,7)]]"
},
        {"name":"multi_hole",
    "outer":"[(0,0),(10,0),(10,10),(0,10)]",
    "holes":"[[(1,1),(2,1),(2,2),(1,2)],[(6,6),(8,6),(8,8), (6,8)]]"}
]

df = pd.DataFrame(data)

df.to_csv("polygon.csv", index = False)

print("CSV generated")

In [ ]:
# CSV to GeoDF
import ast

df = pd.read_csv("polygon.csv")
def create_polygon(row):
    outer = ast.literal_eval(row['outer'])
    holes = ast.literal_eval(row['holes'])
    return Polygon(outer, holes)
df['geometry'] = df.apply(create_polygon, axis = 1)

gdf = gpd.GeoDataFrame(
    df, 
    geometry = "geometry",
    crs = "EPSG:4326"
)
gdf

In [ ]:
print(gdf)

In [ ]:
# plot geoDf

import matplotlib.pyplot as plt

gdf.plot(figsize = (8,6), facecolor = "lightblue", edgecolor = "black")

plt.show()